In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px

zema = ZemaManager()


In [0]:
from sqlmodel import Session, select

In [0]:
import LDCCropMonitor


In [0]:
print(dir(LDCCropMonitor))

In [0]:
import inspect
print(inspect.getmembers(LDCCropMonitor.SoilMoisture))
print(inspect.ismodule(LDCCropMonitor.SoilMoisture))
print(inspect.isclass(LDCCropMonitor.SoilMoisture))

In [0]:
import inspect

inspect.getmembers(LDCCropMonitor.SoilMoisture.SMAPRasterAnalysis, predicate=inspect.isfunction)


In [0]:
from LDCCropMonitor.SoilMoisture import SMAPRasterAnalysis

smap = SMAPRasterAnalysis()


In [0]:
from LDCCropMonitor.SoilMoisture import SMAPRasterAnalysis
import ee
from datetime import datetime
!earthengine authenticate
# Initialize Earth Engine
ee.Initialize()

# Build required arguments
fc = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Argentina'))
band = "ssm"  # Adjust if needed
cycle_start = datetime(2023, 10, 1)
cycle_end = datetime(2024, 9, 30)
crop_year = "2023/2024"
crop_cycle = "main"

# Instantiate SMAPRasterAnalysis
smap = SMAPRasterAnalysis(
    fc=fc,
    band=band,
    cycle_start=cycle_start,
    cycle_end=cycle_end,
    crop_year=crop_year,
    crop_cycle=crop_cycle
)


In [0]:
from LDCCropMonitor.SoilMoisture import SMAPRasterAnalysis

In [0]:
help(SMAPRasterAnalysis)
# or
dir(SMAPRasterAnalysis)

In [0]:
import LDCGeolocation

gid_ldc = LDCGeolocation.create.from_list(
    admin_codes = ["BRA.1", "BRA.2", "BRA.3"],
    name = 'Brazil', # Normally the country name
    platform = 'sugar',
    extra_info = ['west', 'sugarcane']
)
print(gid_ldc)

In [0]:
from sqlmodel import Session
import LDCGeolocation
from LDCCropmonitor import model, database

database.init_dev() # Test your code locally / on dev before pushing into production

ldc_df = LDCGeolocation._administrative.ldc_df()
df = ldc_df[ldc_df.GID_LDC == '<YOUR GID_LDC HERE>'] # Insert the gid_ldc created by the last step

with Session(database.engine) as session:

    # get the associated crop mask
    mask_filter = model.CropMask.name == "<YOUR CROP_MASK HERE>" # Insert your crop mask name here. Default: esa_cropland
    mask_query = select(model.CropMask).where(mask_filter)
    mask = session.exec(mask_query).one()

    # get the commodity from the name
    commodity_filter = model.Commodity.name == "<YOUR COMMODITY HERE>" # You can also get your commodity from the gid_ldc
    commodity_query = select(model.Commodity).where(commodity_filter)
    commodity = session.exec(commodity_query).one()

    # get the geometries
    geom_filter = model.Geometry.gid_code.in_(df["GEOM_CODE"])
    geom_query = select(model.Geometry).filter(geom_filter)
    geometries = session.exec(geom_query).all()

    # create the geolocation
    session.add(
        model.Geolocation(
            gid_ldc=df.iloc[0]["GID_LDC"],
            display_name=df.iloc[0]["NAME_LDC"],
            geometries=geometries,
            crop_mask=mask,
            commodity=commodity,
        )
    )
    session.commit()

In [0]:
from LDCCropMonitor import *

In [0]:

database.init_dev()
# Fetching all data for SoilMoistureI earlier than 2024-12-01
with Session(database.engine) as session:
    query = select(SoilMoistureIEcmwfDaily).where(model.SoilMoistureIEcmwfDaily.date >= "2024-12-01")
    values = session.exec(query).all()

In [0]:
from LDCCropmonitor import database

database.init_prod()